# Import Data

In [9]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random
import warnings
import torch
import multiprocessing
import os


class CFG:
    
    target_name='tag'
    seed = 58
    
    #######################################################################################
    # GPU
    gpu_available = torch.cuda.is_available()

    print(f"CUDA available: {gpu_available}")
    if gpu_available:
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
        print(f"Number of GPUs: {torch.cuda.device_count()}")
    else:
        print(f'Use CPU, \nNumber of CPUs: {multiprocessing.cpu_count()}')
    
    #######################################################################################
    # Seed
    
    @staticmethod
    def seed_all(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


CFG.seed_all(CFG.seed)

warnings.simplefilter('ignore')
sns.set_theme(style="ticks")

CUDA available: False
Use CPU, 
Number of CPUs: 4


In [10]:
x = pd.read_csv('/kaggle/input/datasets/artsmirnovch/spd-feature-set-2/x_train.csv')
y = pd.read_csv('/kaggle/input/datasets/artsmirnovch/spd-feature-set-2/y_train.csv')
x = x.drop('Unnamed: 0', axis=1)
y = y.drop('Unnamed: 0', axis=1)

# Prepare data

In [ ]:
from sklearn.model_selection import train_test_split


x_train, x_val, y_train, y_val = train_test_split(
    x, y, stratify=y, shuffle=True, test_size=0.2, random_state=CFG.seed
)

# Optuna

In [12]:
import optuna

from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def objective(trial):

    if CFG.gpu_available:
        device = "gpu"
        n_jobs_setting = 1
    else:
        device = "cpu"
        n_jobs_setting = -1

    param_space = {
        'boosting_type': 'gbdt',
        'random_state': CFG.seed,
        'n_jobs': n_jobs_setting,   
        'verbose': -1,
        'device_type': device,

        'n_estimators': trial.suggest_int('n_estimators', 100, 3000),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'min_split_gain': trial.suggest_float('min_split_gain', 1e-5, 1e-2, log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-4, 1e-2, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': trial.suggest_int('subsample_freq', 0, 10),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 1.0, log=True),
        'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 10, 200),
    }

    # Var 1
    # local_x_train, local_x_val, local_y_train, local_y_val = train_test_split(
    #     x, y, stratify=y, shuffle=True, test_size=0.25
    # )
    # clf = XGBClassifier(**params).fit(local_x_train, local_y_train, eval_set=[(local_x_val, local_y_val)], verbose=0)
    # y_pred_proba = clf.predict_proba(local_x_val)[:, 1]
    # auc_score = roc_auc_score(local_y_val, y_pred_proba)
    # return {'score': auc_score, 'status': STATUS_OK}
    
    # Var 2
    seed = param_space['random_state'] # solve ModuleNotFoundError
    
    # model = CatBoostClassifier(**params)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    split_generator = skf.split(x_train, y_train)
    
    train_scores = []
    val_scores = []
    for fold, (train_idx, val_idx) in enumerate(split_generator):
        
        fold_x_train = x_train.iloc[train_idx, :]
        fold_y_train = y_train.iloc[train_idx]
        fold_x_val = x_train.iloc[val_idx, :]
        fold_y_val = y_train.iloc[val_idx]
        
        fold_model = LGBMClassifier(**param_space)

        fold_model.fit(
            fold_x_train,
            fold_y_train,
            eval_metric='auc',
            eval_set=[(fold_x_val, fold_y_val)],
        )
        
        fold_train_pred_proba = fold_model.predict_proba(fold_x_train)[:, 1]
        fold_val_pred_proba = fold_model.predict_proba(fold_x_val)[:, 1]
        
        fold_train_score = roc_auc_score(fold_y_train, fold_train_pred_proba)
        fold_val_score = roc_auc_score(fold_y_val, fold_val_pred_proba)
        
        train_scores.append(fold_train_score)
        val_scores.append(fold_val_score)

    train_scores = np.array(train_scores)
    val_scores = np.array(val_scores)

    trial.set_user_attr("train_scores_mean", train_scores.mean())

    # return val_scores.mean() - 4 * abs(val_scores.mean() - train_scores.mean())
    return val_scores.mean()


study = optuna.create_study(
    direction='maximize', 
    study_name='lightgbm_optimization',
    load_if_exists=True
)

study.optimize(objective, n_trials=600, timeout=2*3600, n_jobs=-1, show_progress_bar=True)

[I 2026-02-27 17:52:07,557] A new study created in memory with name: lightgbm_optimization


  0%|          | 0/600 [00:00<?, ?it/s]

[I 2026-02-27 17:53:52,337] Trial 2 finished with value: 0.9571504581202779 and parameters: {'n_estimators': 2885, 'max_depth': 4, 'learning_rate': 0.08787913227756558, 'num_leaves': 49, 'min_split_gain': 0.0006429996220361803, 'min_child_weight': 0.0005464181412647106, 'min_child_samples': 26, 'subsample': 0.9947195113146615, 'subsample_freq': 4, 'colsample_bytree': 0.5557088061716022, 'reg_alpha': 0.06428214727969633, 'reg_lambda': 0.698565276205838, 'early_stopping_rounds': 156}. Best is trial 2 with value: 0.9571504581202779.
[I 2026-02-27 17:55:01,309] Trial 4 finished with value: 0.9563594370052062 and parameters: {'n_estimators': 359, 'max_depth': 5, 'learning_rate': 0.08299751356596048, 'num_leaves': 46, 'min_split_gain': 4.774078694719773e-05, 'min_child_weight': 0.0034013410515391076, 'min_child_samples': 17, 'subsample': 0.5265856670570603, 'subsample_freq': 8, 'colsample_bytree': 0.6729504010181944, 'reg_alpha': 1.0651262312343948e-05, 'reg_lambda': 0.003685566140506453, 'e

In [13]:
best_trial = study.best_trial
print(f"Best validation score: {best_trial.value}")
print(f"Best train score: {best_trial.user_attrs['train_scores_mean']}")

best_params = study.best_params
print("Best parameters:", best_params)

Best validation score: 0.9575252453277452
Best train score: 0.9760725998949458
Best parameters: {'n_estimators': 2787, 'max_depth': 5, 'learning_rate': 0.014355534315226034, 'num_leaves': 74, 'min_split_gain': 3.382364276305923e-05, 'min_child_weight': 0.0006152178074079476, 'min_child_samples': 29, 'subsample': 0.9397569800417903, 'subsample_freq': 10, 'colsample_bytree': 0.5899902119429722, 'reg_alpha': 0.010771395053282081, 'reg_lambda': 0.00021358567374326191, 'early_stopping_rounds': 143}


In [14]:
import plotly.graph_objects as go


trials_df = study.trials_dataframe()
val_scores = trials_df['value'].values
train_scores = [t.user_attrs['train_scores_mean'] for t in study.trials]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(len(val_scores))),
    y=val_scores,
    mode='markers+lines',
    name='Validation Score',
    marker={'color': 'blue'}
))

fig.add_trace(go.Scatter(
    x=list(range(len(train_scores))),
    y=train_scores,
    mode='markers+lines',
    name='Train Score',
    marker={'color': 'red'}
))

fig.update_layout(
    title='Optimization History - Train vs Validation Scores',
    xaxis_title='Trial',
    yaxis_title='AUC Score',
    hovermode='x unified'
)

fig.write_html("optimization_history.html")
fig.show()